Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 27.3 ms, sys: 8.04 ms, total: 35.3 ms
Wall time: 38.2 ms
CPU times: user 12.1 ms, sys: 4.13 ms, total: 16.3 ms
Wall time: 17.8 ms
CPU times: user 4.91 ms, sys: 1.89 ms, total: 6.79 ms
Wall time: 7.71 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 150
alpha = 1
lr = 1e-4
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0
shared_dropout = 0
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.0001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [10]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_method = trial.suggest_categorical("method", ["mmd_rbf", "mmd_linear"])
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            cfr.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "method": t.params["method"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "method": t.params["method"],
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 46. Best value: 0.824535: 100%|██████████| 50/50 [59:57<00:00, 71.94s/it]


In [11]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 46
Best mean_Val_AUQC: 0.824535
Best params: {'lr': 0.0004966073444244392, 'weight_decay': 2.2213037794797527e-05, 'alpha': 0.1, 'method': 'mmd_linear'}

Best config table:


,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,0.000497,0.000022,0.1,mmd_linear,0.824535,0.024963



Top 10 trials:


,trial,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,46,0.000497,0.000022,0.1,mmd_linear,0.824535,0.024963
1,42,0.000498,0.000013,0.1,mmd_rbf,0.823686,0.026939
2,36,0.000497,0.000024,0.1,mmd_rbf,0.823655,0.025311
3,43,0.000487,0.000015,0.1,mmd_rbf,0.823499,0.026810
4,28,0.000499,0.000016,0.1,mmd_rbf,0.823412,0.027037
5,31,0.000499,0.000013,0.1,mmd_rbf,0.822980,0.027585
6,26,0.000492,0.000012,1.0,mmd_rbf,0.822158,0.026841
7,32,0.000493,0.000013,0.1,mmd_rbf,0.822113,0.026906
8,12,0.000458,0.000012,1.0,mmd_rbf,0.821253,0.027343
9,21,0.000466,0.000010,1.0,mmd_rbf,0.821238,0.029021



Per-seed test results:


,Seed,AUUC,AUQC,Lift,KRCC,ATE_Err
0,2,0.555914,0.553094,0.970701,0.038079,0.326113
1,3,0.631428,0.627109,1.371046,0.051196,0.293071
2,4,0.727551,0.724771,0.996727,0.199871,0.425372
3,5,0.603166,0.600046,1.205494,0.034170,0.413682



Test metrics mean ± std:


,AUUC,AUQC,Lift,KRCC,ATE_Err
mean,0.629515,0.626255,1.135992,0.080829,0.364560
std,0.072402,0.072447,0.188677,0.079695,0.065064


In [14]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfr = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=0.000497, 
        weight_decay=2.20E-05, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        method="mmd_linear", alpha=0.1,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfr.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfr.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 444.8240 | Val Loss: 498.3719 | Val Qini: 0.3158 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3158 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 645.9415 | Val Loss: 497.8877 | Val Qini: 0.2534 (patience: 1/20)EMA Trend: 0.3065 | (patience: 1/20)
Epoch 3/150 | Loss: 301.9151 | Val Loss: 497.1699 | Val Qini: 0.2618 (patience: 2/20)EMA Trend: 0.2998 | (patience: 2/20)
Epoch 4/150 | Loss: 431.0866 | Val Loss: 496.4343 | Val Qini: 0.2346 (patience: 3/20)EMA Trend: 0.2900 | (patience: 3/20)
Epoch 5/150 | Loss: 363.9897 | Val Loss: 495.9389 | Val Qini: 0.6362 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3419 | ⭐ NEW BEST (peak ≥ trend)
Ep

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 339.9300 | Val Loss: 498.4050 | Val Qini: 0.3635 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3635 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 605.7828 | Val Loss: 498.0345 | Val Qini: 0.4118 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3708 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 419.9080 | Val Loss: 497.4479 | Val Qini: 0.3661 (patience: 1/20)EMA Trend: 0.3701 | (patience: 1/20)
Epoch 4/150 | Loss: 476.8559 | Val Loss: 496.7403 | Val Qini: 0.3038 (patience: 2/20)EMA Trend: 0.3601 | (patience: 2/20)
Epoch 5/150 | Loss: 476.2428 | Val Loss: 496.1300 | Val Qini: 0.6473 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4032 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 175.2809 | Val Loss: 495.6738 | Val Qini: 0.7398 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4537 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 441.9900 | Val Loss: 495.3761 | Val Qini: 0.7849 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5034 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 343.2751 | Val Loss: 495.2601 | Val 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 289.7181 | Val Loss: 498.7514 | Val Qini: 0.3845 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3845 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 433.3661 | Val Loss: 498.4215 | Val Qini: 0.3683 (patience: 1/20)EMA Trend: 0.3821 | (patience: 1/20)
Epoch 3/150 | Loss: 461.6260 | Val Loss: 497.9124 | Val Qini: 0.4918 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3986 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 339.7700 | Val Loss: 497.1680 | Val Qini: 0.5257 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4176 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 531.4650 | Val Loss: 496.5339 | Val Qini: 0.6045 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4457 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 575.3748 | Val Loss: 496.2438 | Val Qini: 0.6618 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4781 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 479.4435 | Val Loss: 495.9441 | Val Qini: 0.7209 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5145 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 337.4724 | Val Los

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 532.5600 | Val Loss: 498.3814 | Val Qini: 0.2922 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2922 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 238.9988 | Val Loss: 498.0210 | Val Qini: 0.7064 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3543 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 553.5133 | Val Loss: 497.4543 | Val Qini: 0.6998 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4061 | ✓ above trend but not peak (patience: 1/20)
Epoch 4/150 | Loss: 533.4749 | Val Loss: 496.6243 | Val Qini: 0.7279 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4544 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 351.2045 | Val Loss: 495.9933 | Val Qini: 0.7462 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4982 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 297.2454 | Val Loss: 495.7341 | Val Qini: 0.7981 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5432 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 380.7270 | Val Loss: 495.4460 | Val Qini: 0.8394 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5876 | ⭐ NEW BEST

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 301.0057 | Val Loss: 498.4927 | Val Qini: 0.7023 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7023 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 404.8309 | Val Loss: 498.0773 | Val Qini: 0.7815 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7142 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 418.2555 | Val Loss: 497.3531 | Val Qini: 0.8246 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7307 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 550.6932 | Val Loss: 496.2484 | Val Qini: 0.8161 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7435 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: 466.5373 | Val Loss: 495.4853 | Val Qini: 0.8248 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7557 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 502.0025 | Val Loss: 495.4878 | Val Qini: 0.8359 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7677 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 469.2385 | Val Loss: 495.2846 | Val Qini: 0.8398 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7785 | ⭐ NEW BEST

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
